# GLiNER2.5-Decide: every use case in one notebook (CPU/GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/GLiNER2_5-Decide-classification.ipynb)

[GLiNER2.5-Decide](https://huggingface.co/fastino/GLiNER2.5-Decide) is a **340M English classifier** (DeBERTa-v3-large encoder)
from Fastino's GLiNER2.5 family. You pass **any label set at call time** and it scores one or more
decision "heads" in a **single forward pass** — no prompt template, no generated tokens, no fine-tuning.

It is a specialist for **operational decisions**: intent, routing, sentiment, priority, policy, and
multi-label tags. It is *not* a general-purpose LLM: it does not reason, explain, or answer open questions.

**What this notebook covers**

1. Setup and loading
2. Intent: customer support, banking, travel, clinic
3. Sentiment and multi-label product aspects
4. News topic, document type, and book genre
5. Email triage (several heads in one call) and ticket routing
6. Agent gates: handoff to a person, "did the agent finish?"
7. Safety: moderation and spam
8. Operations: incident severity and urgency scores
9. Several decisions at once (single + multi-label heads)
10. Question over a passage (`prompt`)
11. Labels with a description
12. Ordinal scores
13. Confidence scores and thresholds
14. Schema builder API
15. Batching and latency
16. Putting it together: a message router pipeline

> Runs fine on a **free CPU runtime**. A GPU runtime is picked up automatically.

## Setup

`AutoExtractor` (the loader that understands GLiNER2.5 checkpoints) ships in the `[local]` extra of `gliner2`.

In [ ]:
!pip install -q "gliner2[local]"

In [2]:
import json
import time

import torch
from gliner2 import AutoExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

t0 = time.perf_counter()
model = AutoExtractor.from_pretrained("fastino/GLiNER2.5-Decide", map_location=DEVICE)
print(f"loaded in {time.perf_counter() - t0:.1f}s")

device: cpu


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first


loaded in 5.5s


A small helper to run a call and pretty-print the text, the result, and the latency.

In [3]:
def decide(text, schema, **kwargs):
    t0 = time.perf_counter()
    result = model.classify_text(text, schema, **kwargs)
    ms = (time.perf_counter() - t0) * 1000
    preview = text if len(text) <= 160 else text[:157] + "..."
    print(f"TEXT: {preview}")
    print(f"-> {json.dumps(result, indent=2, ensure_ascii=False)}  ({ms:.0f} ms)\n")
    return result

## 1. Intent classification

The label set **is** your product catalog. Change the labels, and the model follows — no retraining.

### Customer support intent
Route a live message before a human ever sees it.

In [4]:
SUPPORT_INTENTS = [
    "order_status", "refund_request", "cancel_subscription", "update_payment",
    "login_problem", "shipping_delay", "bug_report", "speak_to_human", "other",
]

for msg in [
    "My subscription renewed on April 15 for ¥5,400 after the service was already down. Can I get that charge refunded?",
    "I keep getting 'invalid password' even after resetting it twice.",
    "Where is my package? Tracking hasn't updated in 6 days.",
    "Please stop charging me, I want to end my plan at the end of this month.",
]:
    decide(msg, {"intent": SUPPORT_INTENTS})

TEXT: My subscription renewed on April 15 for ¥5,400 after the service was already down. Can I get that charge refunded?
-> {
  "intent": "refund_request"
}  (192 ms)

TEXT: I keep getting 'invalid password' even after resetting it twice.
-> {
  "intent": "login_problem"
}  (149 ms)



TEXT: Where is my package? Tracking hasn't updated in 6 days.
-> {
  "intent": "order_status"
}  (168 ms)

TEXT: Please stop charging me, I want to end my plan at the end of this month.
-> {
  "intent": "cancel_subscription"
}  (153 ms)



### Banking request

In [5]:
decide(
    "The transfer I sent this morning is still pending, and I think I used the wrong sort code. "
    "Can you stop it and add Emily as the beneficiary instead?",
    {"intent": [
        "transfer_pending", "transfer_cancel", "beneficiary_add", "card_lost",
        "balance_inquiry", "fraud_report", "mortgage_application", "fee_explanation",
    ]},
)

TEXT: The transfer I sent this morning is still pending, and I think I used the wrong sort code. Can you stop it and add Emily as the beneficiary instead?
-> {
  "intent": "beneficiary_add"
}  (157 ms)



{'intent': 'beneficiary_add'}

### Travel request

In [6]:
decide(
    "I need to move my Friday flight to Paris to Saturday morning, same cabin, and keep the aisle seat if you can.",
    {"request": ["book", "change", "cancel", "status", "seat_change", "refund", "baggage"]},
)

TEXT: I need to move my Friday flight to Paris to Saturday morning, same cabin, and keep the aisle seat if you can.
-> {
  "request": "seat_change"
}  (143 ms)



{'request': 'seat_change'}

### Clinic request

In [7]:
decide(
    "The rash came back after the antibiotics finished. Can I get a same-week appointment with dermatology, "
    "or should I just refill the cream?",
    {"request": [
        "book_appointment", "refill_prescription", "test_results",
        "referral", "billing_question", "cancel_appointment",
    ]},
)

TEXT: The rash came back after the antibiotics finished. Can I get a same-week appointment with dermatology, or should I just refill the cream?
-> {
  "request": "book_appointment"
}  (168 ms)



{'request': 'book_appointment'}

## 2. Sentiment and product aspects

A single-label head gives the overall sentiment. A **multi-label** head (`multi_label=True`) returns every
label above `cls_threshold` — here, *which* parts of the product the review is about.

In [8]:
review = "Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop."

decide(review, {"sentiment": ["positive", "negative", "mixed", "neutral"]})

decide(review, {"aspects": {
    "labels": ["battery", "keyboard", "screen", "camera", "price", "support"],
    "multi_label": True,
    "cls_threshold": 0.4,
}})

TEXT: Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.
-> {
  "sentiment": "positive"
}  (138 ms)



TEXT: Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.
-> {
  "aspects": [
    "battery",
    "keyboard",
    "screen"
  ]
}  (142 ms)



{'aspects': ['battery', 'keyboard', 'screen']}

## 3. Topic, document type, and genre

### News topic

In [9]:
decide(
    "The central bank held rates and said inflation is still above target, pushing bank stocks lower in afternoon trading.",
    {"topic": ["politics", "business", "sports", "science", "entertainment", "world"]},
)

TEXT: The central bank held rates and said inflation is still above target, pushing bank stocks lower in afternoon trading.
-> {
  "topic": "business"
}  (142 ms)



{'topic': 'business'}

### Document type
Classifying the document is the gate in front of extraction.

In [10]:
DOC_TYPES = ["invoice", "receipt", "contract", "resume", "support_email", "meeting_notes"]

decide(
    "INVOICE 1842\nBill to: Northstar QA\nAmount due: 2,400 USD\nDue: 30 April 2026\nWire instructions are on page 2.",
    {"document_type": DOC_TYPES},
)
decide(
    "Attendees: Ana, Raj, Lee\nAction items: Raj to send the Q3 forecast by Tuesday. Next sync on Thursday.",
    {"document_type": DOC_TYPES},
)

TEXT: INVOICE 1842
Bill to: Northstar QA
Amount due: 2,400 USD
Due: 30 April 2026
Wire instructions are on page 2.
-> {
  "document_type": "invoice"
}  (145 ms)



TEXT: Attendees: Ana, Raj, Lee
Action items: Raj to send the Q3 forecast by Tuesday. Next sync on Thursday.
-> {
  "document_type": "meeting_notes"
}  (174 ms)



{'document_type': 'meeting_notes'}

### Book genre

In [11]:
decide(
    "She closed the ledger, blew out the lamp, and listened for the stair. "
    "The house had been empty since the winter the river took the bridge.",
    {"genre": ["mystery", "romance", "history", "science_fiction", "literary_fiction", "cookbook"]},
)

TEXT: She closed the ledger, blew out the lamp, and listened for the stair. The house had been empty since the winter the river took the bridge.


-> {
  "genre": "history"
}  (184 ms)



{'genre': 'history'}

## 4. Email triage and ticket routing

### Email triage — three heads, one call
Pass several tasks in the same schema dict. They are all scored in one forward pass.

In [12]:
decide(
    "From: compliance@group.example\nSubject: Protocol update — action required today\n\n"
    "Please confirm the new retention rule is applied before Friday's audit.",
    {
        "intent": ["fyi", "request", "approval", "complaint", "newsletter", "security_alert"],
        "urgency": ["low", "normal", "high", "critical"],
        "route": ["support", "billing", "legal", "security", "finance", "archive"],
    },
)

TEXT: From: compliance@group.example
Subject: Protocol update — action required today

Please confirm the new retention rule is applied before Friday's audit.
-> {
  "intent": "request",
  "urgency": "critical",
  "route": "legal"
}  (160 ms)



{'intent': 'request', 'urgency': 'critical', 'route': 'legal'}

### Ticket routing

In [13]:
decide(
    "[subject] 401k deduction missing from this paystub\n"
    "[body] Last month's contribution posted. This month the line is gone and HR told me to open a ticket.",
    {"queue": [
        "payroll", "benefits", "it_access", "facilities",
        "expense_reimbursement", "manager_approval",
    ]},
)

TEXT: [subject] 401k deduction missing from this paystub
[body] Last month's contribution posted. This month the line is gone and HR told me to open a ticket.
-> {
  "queue": "benefits"
}  (139 ms)



{'queue': 'benefits'}

## 5. Agent gates

Binary `yes`/`no` heads make cheap guards around LLM agents.

### Handoff to a person

With bare `yes`/`no` labels, the **task name and prompt carry the meaning**. A terse name like `handoff`
is ambiguous; a descriptive name (`needs_human`) or an explicit `prompt` makes the question clear.

In [14]:
HANDOFF_MSGS = [
    "This is the third time I have explained the same missing refund. Stop the bot and get me a person.",
    "Thanks, that fixed it!",
]

for msg in HANDOFF_MSGS:
    bare = model.classify_text(msg, {"handoff": ["yes", "no"]})["handoff"]
    named = model.classify_text(msg, {"needs_human": ["yes", "no"]})["needs_human"]
    prompted = model.classify_text(msg, {"handoff": {
        "labels": ["yes", "no"],
        "prompt": "Should this conversation be handed off to a human agent?",
    }})["handoff"]
    print(f"{msg[:60]:<62} bare={bare:<4} needs_human={named:<4} prompt={prompted}")

This is the third time I have explained the same missing ref   bare=no   needs_human=yes  prompt=yes


Thanks, that fixed it!                                         bare=yes  needs_human=no   prompt=no


### Did the agent finish?
Judge completion from the goal plus the last state, not from whether the agent stopped talking.

In [15]:
decide(
    "Goal: email the Q4 summary to every partner.\nLast action: draft saved in the hub.\n"
    "Send button is still disabled because two partners have no address.",
    {"finished": ["yes", "no"]},
)
decide(
    "Goal: email the Q4 summary to every partner.\nLast action: sent to all 12 partners.\n"
    "Delivery report shows 12/12 delivered.",
    {"finished": ["yes", "no"]},
)

TEXT: Goal: email the Q4 summary to every partner.
Last action: draft saved in the hub.
Send button is still disabled because two partners have no address.


-> {
  "finished": "no"
}  (171 ms)

TEXT: Goal: email the Q4 summary to every partner.
Last action: sent to all 12 partners.
Delivery report shows 12/12 delivered.
-> {
  "finished": "yes"
}  (129 ms)



{'finished': 'yes'}

## 6. Safety: moderation and spam

In [16]:
POLICY = ["allow", "personal_data", "harassment", "scam", "violence", "spam"]

decide(
    "Post the customer's home address in the public thread so everyone can see where the package actually went.",
    {"policy": POLICY},
)
decide("Can you share the tracking link for order 5521?", {"policy": POLICY})

TEXT: Post the customer's home address in the public thread so everyone can see where the package actually went.
-> {
  "policy": "personal_data"
}  (152 ms)

TEXT: Can you share the tracking link for order 5521?
-> {
  "policy": "allow"
}  (143 ms)



{'policy': 'allow'}

In [17]:
for msg in [
    "Your mailbox is almost full. Click here in the next hour or we will delete every message.",
    "Hi team, attaching the slides for tomorrow's review. Let me know if anything is missing.",
]:
    decide(msg, {"label": ["spam", "ham"]})

TEXT: Your mailbox is almost full. Click here in the next hour or we will delete every message.
-> {
  "label": "spam"
}  (128 ms)

TEXT: Hi team, attaching the slides for tomorrow's review. Let me know if anything is missing.
-> {
  "label": "ham"
}  (131 ms)



## 7. Operations: severity and urgency

### Incident severity

In [18]:
SEVERITY = ["info", "low", "medium", "high", "critical"]

decide(
    "The deploy left resource tags inconsistent across staging. Production checkout is unaffected. No customer reports yet.",
    {"severity": SEVERITY},
)
decide(
    "Checkout is returning 500s for all users in EU since 14:02. Revenue dashboard is flat.",
    {"severity": SEVERITY},
)

TEXT: The deploy left resource tags inconsistent across staging. Production checkout is unaffected. No customer reports yet.
-> {
  "severity": "info"
}  (131 ms)

TEXT: Checkout is returning 500s for all users in EU since 14:02. Revenue dashboard is flat.
-> {
  "severity": "critical"
}  (128 ms)



{'severity': 'critical'}

### Urgency score

Pass `'0'` to `'5'` as ordinary string labels to get a rank.

In [19]:
URGENCY = ["0", "1", "2", "3", "4", "5"]

decide("Payroll file has to be corrected before the 5pm cutoff or the whole company is paid late.", {"urgency": URGENCY})
decide("There's a typo in the onboarding wiki page, whenever someone has time.", {"urgency": URGENCY})

TEXT: Payroll file has to be corrected before the 5pm cutoff or the whole company is paid late.
-> {
  "urgency": "5"
}  (132 ms)

TEXT: There's a typo in the onboarding wiki page, whenever someone has time.
-> {
  "urgency": "0"
}  (137 ms)



{'urgency': '0'}

## 8. Several decisions at once

Mix single-label and multi-label heads in one schema. This is how a property system opens the right
work orders without a chain of prompts.

In [20]:
decide(
    "Guest in room 1408 says the AC has been out since yesterday and they want to move tonight or leave. "
    "They also asked for the incidentals hold to be released.",
    {
        "intent": ["maintenance", "room_change", "checkout", "billing", "complaint", "amenity_request"],
        "priority": ["low", "normal", "high", "urgent"],
        "needs_human": ["yes", "no"],
        "topics": {
            "labels": ["hvac", "billing", "housekeeping", "noise", "safety"],
            "multi_label": True,
            "cls_threshold": 0.4,
        },
    },
)

TEXT: Guest in room 1408 says the AC has been out since yesterday and they want to move tonight or leave. They also asked for the incidentals hold to be released.
-> {
  "intent": "room_change",
  "priority": "urgent",
  "needs_human": "yes",
  "topics": [
    "hvac"
  ]
}  (177 ms)



{'intent': 'room_change',
 'priority': 'urgent',
 'needs_human': 'yes',
 'topics': ['hvac']}

## 9. Question over a passage

Add a `prompt` to a task: the text is the source, the prompt is the question, and the label is the answer.

In [21]:
passage = (
    "The treaty was signed in Paris in 1992. It entered into force the following year, "
    "after the last signatory ratified it."
)

for q in [
    "Did the treaty enter into force in 1992?",
    "Was the treaty signed in Paris?",
]:
    print("Q:", q)
    decide(passage, {"answer": {"labels": ["yes", "no"], "prompt": q}})

Q: Did the treaty enter into force in 1992?
TEXT: The treaty was signed in Paris in 1992. It entered into force the following year, after the last signatory ratified it.
-> {
  "answer": "yes"
}  (150 ms)

Q: Was the treaty signed in Paris?


TEXT: The treaty was signed in Paris in 1992. It entered into force the following year, after the last signatory ratified it.
-> {
  "answer": "yes"
}  (140 ms)



> Passage QA is the weakest skill here: questions that need arithmetic or inference over the text
> ("the following year" = 1993) can come back wrong. Keep it for direct lookups and use an LLM when the
> answer needs reasoning.

## 10. Labels with a description

When a label name is ambiguous, pass a `{label: description}` dict. The description becomes part of the decision.

In [22]:
decide(
    "Please reset the card PIN. The new one never arrived and the old one is locked after three tries.",
    {"intent": {
        "labels": {
            "card_pin_change": "The customer wants a new PIN or the current PIN replaced",
            "card_lost": "The physical card is missing",
            "balance_inquiry": "The customer wants the current balance",
        },
    }},
)

TEXT: Please reset the card PIN. The new one never arrived and the old one is locked after three tries.
-> {
  "intent": "card_pin_change"
}  (151 ms)



{'intent': 'card_pin_change'}

## 11. Ordinal score (0–10)

In [23]:
RATING = [str(i) for i in range(11)]

for r in [
    "I finished it in two nights. The ending is earned, the middle drags, and I would still hand it to a friend.",
    "Gave up after 40 pages. Flat characters and a plot you can see coming from the cover.",
]:
    decide(r, {"rating": RATING})

TEXT: I finished it in two nights. The ending is earned, the middle drags, and I would still hand it to a friend.
-> {
  "rating": "7"
}  (141 ms)



TEXT: Gave up after 40 pages. Flat characters and a plot you can see coming from the cover.
-> {
  "rating": "0"
}  (161 ms)



## 12. Confidence scores and thresholds

`include_confidence=True` returns `{"label": ..., "confidence": ...}` instead of a bare string — useful
for abstaining or escalating when the model is unsure. For multi-label heads, `cls_threshold` controls how
many labels come back.

In [24]:
decide(
    "Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.",
    {"sentiment": ["positive", "negative", "mixed", "neutral"]},
    include_confidence=True,
)

TEXT: Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.
-> {
  "sentiment": {
    "label": "positive",
    "confidence": 0.9984742999076843
  }
}  (131 ms)



{'sentiment': {'label': 'positive', 'confidence': 0.9984742999076843}}

In [25]:
text = "Loved the camera and the price was fair, but support never answered my emails about the screen flicker."
ASPECTS = ["battery", "keyboard", "screen", "camera", "price", "support"]

for th in [0.2, 0.4, 0.6, 0.8]:
    res = model.classify_text(
        text,
        {"aspects": {"labels": ASPECTS, "multi_label": True, "cls_threshold": th}},
        include_confidence=True,
    )
    print(f"threshold={th}: {res['aspects']}")

threshold=0.2: [{'label': 'screen', 'confidence': 0.6008234620094299}, {'label': 'camera', 'confidence': 0.8944309949874878}, {'label': 'price', 'confidence': 0.8144071698188782}, {'label': 'support', 'confidence': 0.8960593342781067}]


threshold=0.4: [{'label': 'screen', 'confidence': 0.6008234620094299}, {'label': 'camera', 'confidence': 0.8944309949874878}, {'label': 'price', 'confidence': 0.8144071698188782}, {'label': 'support', 'confidence': 0.8960593342781067}]


threshold=0.6: [{'label': 'screen', 'confidence': 0.6008234620094299}, {'label': 'camera', 'confidence': 0.8944309949874878}, {'label': 'price', 'confidence': 0.8144071698188782}, {'label': 'support', 'confidence': 0.8960593342781067}]


threshold=0.8: [{'label': 'camera', 'confidence': 0.8944309949874878}, {'label': 'price', 'confidence': 0.8144071698188782}, {'label': 'support', 'confidence': 0.8960593342781067}]


A simple abstain rule: only auto-route when confidence clears a bar. With many labels the score is
spread across candidates, so calibrate `min_conf` on your own data.

In [26]:
def route_or_escalate(text, labels, min_conf=0.35):
    res = model.classify_text(text, {"intent": labels}, include_confidence=True)["intent"]
    if res["confidence"] >= min_conf:
        return f"auto-route -> {res['label']} ({res['confidence']:.2f})"
    return f"escalate to human (best guess {res['label']} @ {res['confidence']:.2f})"

for msg in [
    "Where is my package? Tracking hasn't updated in 6 days.",
    "hmm not sure, something is off with my account I think",
]:
    print(msg, "\n  ", route_or_escalate(msg, SUPPORT_INTENTS))

Where is my package? Tracking hasn't updated in 6 days. 
   auto-route -> order_status (0.39)


hmm not sure, something is off with my account I think 
   escalate to human (best guess other @ 0.26)


## 13. Schema builder API

The dict schema is shorthand. The builder API expresses the same thing and is handy when you assemble
heads programmatically.

In [27]:
schema = (
    model.create_schema()
    .classification("sentiment", ["positive", "negative", "mixed", "neutral"])
    .classification(
        "aspects",
        ["battery", "keyboard", "screen", "camera", "price", "support"],
        multi_label=True,
        cls_threshold=0.4,
    )
)

print(model.extract(
    "Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.",
    schema,
    include_confidence=True,
))

{'sentiment': {'label': 'positive', 'confidence': 0.9994935989379883}, 'aspects': [{'label': 'battery', 'confidence': 0.7610358595848083}, {'label': 'keyboard', 'confidence': 0.9939987659454346}, {'label': 'screen', 'confidence': 0.9846210479736328}]}


## 14. Batching and latency

Every call is a single forward pass, so adding heads costs far less than calling the model once per head.

In [28]:
tickets = [
    "My subscription renewed after the service was already down. Refund please.",
    "I keep getting 'invalid password' even after resetting it twice.",
    "Where is my package? Tracking hasn't updated in 6 days.",
    "The app crashes every time I open settings on Android 14.",
    "I want to talk to a real person, not a bot.",
    "Please update the card on file, the old one expired.",
    "Cancel my plan at the end of this billing cycle.",
    "Order #9912 — when will it ship?",
] * 4

schema = {"intent": SUPPORT_INTENTS}
model.classify_text(tickets[0], schema)  # warm-up

t0 = time.perf_counter()
preds = [model.classify_text(t, schema)["intent"] for t in tickets]
elapsed = time.perf_counter() - t0
print(f"{len(tickets)} texts in {elapsed:.2f}s -> {elapsed / len(tickets) * 1000:.0f} ms/text on {DEVICE}\n")
for t, p in list(zip(tickets, preds))[:8]:
    print(f"{p:<20} {t}")

32 texts in 5.22s -> 163 ms/text on cpu

refund_request       My subscription renewed after the service was already down. Refund please.
login_problem        I keep getting 'invalid password' even after resetting it twice.
order_status         Where is my package? Tracking hasn't updated in 6 days.
bug_report           The app crashes every time I open settings on Android 14.
speak_to_human       I want to talk to a real person, not a bot.
update_payment       Please update the card on file, the old one expired.
cancel_subscription  Cancel my plan at the end of this billing cycle.
shipping_delay       Order #9912 — when will it ship?


In [29]:
multi_head = {
    "intent": SUPPORT_INTENTS,
    "sentiment": ["positive", "negative", "mixed", "neutral"],
    "urgency": ["low", "normal", "high", "critical"],
    "handoff": ["yes", "no"],
}

for name, s in [("1 head", schema), ("4 heads", multi_head)]:
    t0 = time.perf_counter()
    for t in tickets:
        model.classify_text(t, s)
    print(f"{name}: {(time.perf_counter() - t0) / len(tickets) * 1000:.0f} ms/text")

1 head: 188 ms/text


4 heads: 158 ms/text


On a GPU you can also enable fp16 and `torch.compile`:

```python
model = AutoExtractor.from_pretrained("fastino/GLiNER2.5-Decide", map_location="cuda", quantize=True, compile=True)
```

## 15. Putting it together: a message router

Chain cheap gates the way a production inbox would: **policy (spam, PII, abuse) → handoff + intent + urgency**.
Each step is one small classifier call; nothing is generated.

In [30]:
def route(message):
    policy = model.classify_text(message, {"policy": POLICY})["policy"]
    if policy != "allow":
        return {"action": "drop" if policy == "spam" else "block", "reason": policy}

    out = model.classify_text(message, {
        "handoff": {
            "labels": ["yes", "no"],
            "prompt": "Should this conversation be handed off to a human agent?",
        },
        "intent": SUPPORT_INTENTS,
        "urgency": ["low", "normal", "high", "critical"],
    })
    if out["handoff"] == "yes" or out["intent"] == "speak_to_human":
        return {"action": "human_queue", **out}
    return {"action": f"workflow:{out['intent']}", **out}

inbox = [
    "Your mailbox is almost full. Click here in the next hour or we will delete every message.",
    "Post the customer's home address in the public thread so everyone knows where they live.",
    "This is the third time I have explained the same missing refund. Get me a person.",
    "I was double charged this morning, please refund one of the payments.",
    "Tracking for my order hasn't moved since Monday.",
]
for m in inbox:
    print(f"{m[:70]:<72} -> {route(m)}")

Your mailbox is almost full. Click here in the next hour or we will de   -> {'action': 'drop', 'reason': 'spam'}


Post the customer's home address in the public thread so everyone know   -> {'action': 'block', 'reason': 'personal_data'}


This is the third time I have explained the same missing refund. Get m   -> {'action': 'human_queue', 'handoff': 'yes', 'intent': 'refund_request', 'urgency': 'high'}


I was double charged this morning, please refund one of the payments.    -> {'action': 'workflow:refund_request', 'handoff': 'no', 'intent': 'refund_request', 'urgency': 'normal'}


Tracking for my order hasn't moved since Monday.                         -> {'action': 'workflow:order_status', 'handoff': 'no', 'intent': 'order_status', 'urgency': 'low'}


## Notes

- **Be descriptive.** Task names, label names, label descriptions, and `prompt` are all part of the input.
  If a head misbehaves, rename it or add a description before reaching for a bigger model.
- **Mixed sentiment and passage QA** are the shakiest heads in this notebook; check them on your own data.
- **English only.** Use [`GLiNER2.5-multi-Decide`](https://huggingface.co/fastino/GLiNER2.5-multi-Decide) for multilingual input.
- **Bigger sibling:** [`GLiNER2.5-Decide-1B`](https://huggingface.co/fastino/GLiNER2.5-Decide-1B).
- **Benchmark:** 60.2% avg exact-match on [`fastino/fast-decisions`](https://huggingface.co/datasets/fastino/fast-decisions) (17 domains).
- **License:** Apache 2.0. Library: [fastino-ai/GLiNER2](https://github.com/fastino-ai/GLiNER2). Paper: [arXiv:2507.18546](https://arxiv.org/abs/2507.18546).